# 22 · GSE135779 · scRNA_seq · pseudobulk

Reads the per-donor count matrices and `metadata.rds`. Writes `data/run_artifacts/GSE135779/pseudobulk_counts.rds`.

**Pseudobulk.** For each child, the UMI counts of every cell in that child's matrix are summed per
gene, giving one column per child, like a bulk sample of that child's PBMC. No cell-type labels are
used (GEO provides none). The matrices share one gene list, `GSE135779_genes.tsv.gz` (Ensembl
identifier and symbol).

In [1]:
source("../src/paths.R")
suppressMessages(library(Matrix))
meta  <- readRDS(art("GSE135779", "metadata.rds"))
genes <- read.delim(raw("GSE135779", "GSE135779_genes.tsv.gz"), header = FALSE, col.names = c("ensembl", "symbol"))
files <- list.files(raw("GSE135779", "raw"), pattern = "matrix.mtx.gz$", full.names = TRUE)
files <- setNames(files, sub("_.*", "", basename(files)))
stopifnot(all(rownames(meta) %in% names(files)))
c(genes = nrow(genes), children = nrow(meta))

genes children 
   32738       44

In [2]:
cells <- setNames(integer(nrow(meta)), rownames(meta))
counts <- sapply(rownames(meta), function(g) {
  m <- readMM(gzfile(files[[g]]))
  stopifnot(nrow(m) == nrow(genes))
  cells[g] <<- ncol(m)
  rowSums(m)
})
rownames(counts) <- genes$ensembl
stopifnot(all(counts == round(counts)))
meta$cells <- cells[rownames(meta)]
data.frame(donor = meta$donor, sle = meta$sle, cells = meta$cells, total_counts_millions = round(colSums(counts) / 1e6, 1))

,donor,sle,cells,total_counts_millions
,<chr>,<dbl>,<int>,<dbl>
GSM4029896,cSLE1,1,4644,12.1
GSM4029897,cSLE2,1,5205,13.3
GSM4029898,cSLE3,1,3894,13.2
GSM4029899,cSLE4,1,6171,18.0
GSM4029900,cSLE5,1,4234,13.0
GSM4029901,cSLE6,1,5438,16.9
GSM4029902,cSLE7,1,3974,12.2
GSM4029903,cSLE8,1,6260,20.7
GSM4029904,cSLE10,1,13538,51.7


**Result.** 32,738 genes. Each child contributes 2,965 to 13,834 cells (median 6,031), and 9.0 to
53.9 million counts in total.

In [3]:
saveRDS(list(counts = counts, genes = genes, meta = meta), art("GSE135779", "pseudobulk_counts.rds"))